In [ ]:
from manim import *
import numpy as np

class GradientConcept(ThreeDScene):
    def construct(self):
        # --- CONFIGURATION ---
        self.camera.background_color = "#1e1e1e"
        
        # --- 2D HUD SETUP (Fixed in Frame) ---
        # 1. Title
        title = Title("Gradient of a Function $f(x,y)$")
        self.add_fixed_in_frame_mobjects(title)
        
        # 2. Formula & Function (Top Left)
        # We start from the Top Left (UL) and stack downwards
        formula = MathTex(
            r"\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix} = \frac{\partial f}{\partial x}\hat{i} + \frac{\partial f}{\partial y}\hat{j}"
        ).scale(0.7)
        formula.to_corner(UL).shift(DOWN * 1.0 + RIGHT * 0.5)
        
        func_eq = MathTex(
            r"f(x,y) = 3 - 0.3(x^2 + y^2)", 
            color=BLUE_B
        ).scale(0.7).next_to(formula, DOWN, buff=0.3).align_to(formula, LEFT)
        
        desc_text = Tex(
            r"The gradient vector points in the\\direction of steepest ascent.",
            font_size=24
        ).next_to(func_eq, DOWN, buff=0.3).align_to(formula, LEFT)

        # 3. Value Display (The Dashboard)
        # FIX: Moved to Top Right (UR) as requested
        # We shift LEFT significantly to leave room for the numbers on the right
        val_label_x = MathTex(r"\frac{\partial f}{\partial x} =", color=RED).scale(0.8)
        val_label_x.to_corner(UR).shift(DOWN * 1.2 + LEFT * 3.0)
        
        val_label_y = MathTex(r"\frac{\partial f}{\partial y} =", color=GREEN).scale(0.8)
        val_label_y.next_to(val_label_x, DOWN, buff=0.4).align_to(val_label_x, LEFT)
        
        # Create Numbers
        val_num_x = DecimalNumber(0, num_decimal_places=2, color=RED).scale(0.8)
        val_num_y = DecimalNumber(0, num_decimal_places=2, color=GREEN).scale(0.8)
        
        # Position numbers to the right of labels
        val_num_x.next_to(val_label_x, RIGHT, buff=0.5)
        val_num_y.next_to(val_label_y, RIGHT, buff=0.5)
        
        val_num_x.set_value(-0.90)
        val_num_y.set_value(-0.60)
        
        # HIDE INITIALLY: Set opacity to 0
        val_label_x.set_opacity(0)
        val_label_y.set_opacity(0)
        val_num_x.set_opacity(0)
        val_num_y.set_opacity(0)
        
        # Add to fixed frame (invisible for now, but position is locked)
        self.add_fixed_in_frame_mobjects(
            formula, func_eq, desc_text, 
            val_label_x, val_label_y, val_num_x, val_num_y
        )

        # --- 3D SCENE SETUP ---
        axes = ThreeDAxes(
            x_range=[-3, 3, 1],
            y_range=[-3, 3, 1],
            z_range=[-2, 4, 1],
            x_length=6,
            y_length=6,
            z_length=5
        ).shift(RIGHT * 3.0 + UP * 0.5) 
        
        x_label = axes.get_x_axis_label("x")
        y_label = axes.get_y_axis_label("y")
        z_label = axes.get_z_axis_label("f(x,y)")
        
        def func(u, v):
            return 3 - 0.3 * (u**2 + v**2)

        surface = Surface(
            lambda u, v: axes.c2p(u, v, func(u, v)),
            u_range=[-3, 3],
            v_range=[-3, 3],
            resolution=(20, 20),
            should_make_jagged=False
        )
        surface.set_style(fill_opacity=0.6, stroke_color=BLUE_E, stroke_width=0.5)
        surface.set_fill_by_checkerboard(BLUE_D, BLUE_E, opacity=0.5)

        self.set_camera_orientation(phi=60 * DEGREES, theta=-45 * DEGREES, zoom=0.7)
        self.add(axes, x_label, y_label, z_label, surface)
        self.wait(1)

        # --- PART 1: PARTIAL X COMPONENT ---
        
        # Step text anchors to Bottom Left (DL)
        step1_text = Tex(
            r"1. Component along $x$: $\frac{\partial f}{\partial x}$",
            color=RED
        ).scale(0.7)
        step1_text.to_corner(DL).shift(UP * 1.5 + RIGHT * 0.5)
        
        self.add_fixed_in_frame_mobjects(step1_text) 
        self.play(FadeIn(step1_text)) 

        # Setup 3D Point
        x0, y0 = 1.5, 1.0
        z0 = func(x0, y0)
        p_loc = axes.c2p(x0, y0, z0)
        dot_p = Dot3D(point=p_loc, color=YELLOW, radius=0.1)
        
        self.move_camera(phi=80 * DEGREES, theta=-90 * DEGREES, run_time=2)
        
        x_curve = ParametricFunction(
            lambda t: axes.c2p(t, y0, func(t, y0)),
            t_range=[-3, 3], color=RED_B
        )
        self.play(Create(x_curve), FadeIn(dot_p))
        
        h_tracker = ValueTracker(1.0) 
        
        def get_secant_elements():
            h = h_tracker.get_value()
            x2 = x0 - h 
            z2 = func(x2, y0)
            
            p1 = axes.c2p(x0, y0, z0)
            p2 = axes.c2p(x2, y0, z2)
            
            sec_line = Line3D(start=p1, end=p2, color=RED)
            line_z1 = DashedLine(p1, axes.c2p(0, y0, z0), color=GRAY, dash_length=0.1)
            line_z2 = DashedLine(p2, axes.c2p(0, y0, z2), color=GRAY, dash_length=0.1)
            line_x1 = DashedLine(p1, axes.c2p(x0, y0, 0), color=GRAY, dash_length=0.1)
            line_x2 = DashedLine(p2, axes.c2p(x2, y0, 0), color=GRAY, dash_length=0.1)
            
            return VGroup(sec_line, line_z1, line_z2, line_x1, line_x2)

        secant_group = always_redraw(get_secant_elements)
        self.add(secant_group)
        self.wait(0.5)
        
        slope_text = MathTex(
            r"\approx \frac{f(x_2) - f(x_1)}{x_2 - x_1}", color=RED
        ).scale(0.7).next_to(step1_text, DOWN).align_to(step1_text, LEFT)
        self.add_fixed_in_frame_mobjects(slope_text)

        self.move_camera(frame_center=axes.c2p(x0, y0, z0), zoom=2.5, run_time=2)
        
        self.play(h_tracker.animate.set_value(0.01), run_time=3)
        self.remove(secant_group) 
        
        self.move_camera(frame_center=ORIGIN, zoom=0.7, run_time=1.5)

        df_dx = -0.6 * x0
        dz_x_vis = df_dx * (-0.6 * x0) 
        
        arrow_x_3d = Arrow(
            start=axes.c2p(x0, y0, z0), 
            end=axes.c2p(x0 + df_dx, y0, z0 + dz_x_vis), 
            color=RED, buff=0, stroke_width=3
        )
        self.play(Create(arrow_x_3d))
        self.wait()
        
        self.play(FadeOut(step1_text), FadeOut(slope_text))

        # --- PART 2: PARTIAL Y COMPONENT ---
        
        step2_text = Tex(
            r"2. Component along $y$: $\frac{\partial f}{\partial y}$",
            color=GREEN
        ).scale(0.7)
        # Anchor to same bottom position
        step2_text.to_corner(DL).shift(UP * 1.5 + RIGHT * 0.5)
        
        self.add_fixed_in_frame_mobjects(step2_text)
        self.play(FadeIn(step2_text))

        self.move_camera(phi=80 * DEGREES, theta=0 * DEGREES, run_time=2)
        self.play(FadeOut(x_curve))

        y_curve = ParametricFunction(
            lambda t: axes.c2p(x0, t, func(x0, t)),
            t_range=[-3, 3], color=GREEN_B
        )
        self.play(Create(y_curve))

        h_tracker.set_value(1.0)
        
        def get_y_secant_elements():
            h = h_tracker.get_value()
            y2 = y0 - h 
            z2 = func(x0, y2)
            
            p1 = axes.c2p(x0, y0, z0)
            p2 = axes.c2p(x0, y2, z2)
            
            sec_line = Line3D(start=p1, end=p2, color=GREEN)
            line_z1 = DashedLine(p1, axes.c2p(x0, 0, z0), color=GRAY, dash_length=0.1)
            line_z2 = DashedLine(p2, axes.c2p(x0, 0, z2), color=GRAY, dash_length=0.1)
            line_y1 = DashedLine(p1, axes.c2p(x0, y0, 0), color=GRAY, dash_length=0.1)
            line_y2 = DashedLine(p2, axes.c2p(x0, y2, 0), color=GRAY, dash_length=0.1)
            
            return VGroup(sec_line, line_z1, line_z2, line_y1, line_y2)

        secant_y_group = always_redraw(get_y_secant_elements)
        self.add(secant_y_group)
        self.wait(0.5)
        
        self.move_camera(frame_center=axes.c2p(x0, y0, z0), zoom=2.5, run_time=2)
        
        self.play(h_tracker.animate.set_value(0.01), run_time=3)
        self.remove(secant_y_group)
        
        self.move_camera(frame_center=ORIGIN, zoom=0.7, run_time=1.5)

        df_dy = -0.6 * y0
        dz_y_vis = df_dy * (-0.6 * y0)
        
        arrow_y_3d = Arrow(
            start=axes.c2p(x0, y0, z0), 
            end=axes.c2p(x0, y0 + df_dy, z0 + dz_y_vis), 
            color=GREEN, buff=0, stroke_width=3
        )
        self.play(Create(arrow_y_3d), FadeOut(y_curve))
        
        self.play(FadeOut(step2_text))

        # --- PART 3: RESULTANT GRADIENT ---
        
        step3_text = Tex(
            r"3. Resultant Gradient Vector",
            color=YELLOW
        ).scale(0.7)
        step3_text.to_corner(DL).shift(UP * 1.5 + RIGHT * 0.5)
        
        self.add_fixed_in_frame_mobjects(step3_text)
        self.play(FadeIn(step3_text))

        self.move_camera(phi=40 * DEGREES, theta=-45 * DEGREES, run_time=2)

        vec_x_end = axes.c2p(x0 + df_dx, y0, z0 + dz_x_vis)
        vec_y_end = axes.c2p(x0, y0 + df_dy, z0 + dz_y_vis)
        
        dz_total = dz_x_vis + dz_y_vis
        vec_grad_end = axes.c2p(x0 + df_dx, y0 + df_dy, z0 + dz_total)
        
        para_line1 = DashedLine(vec_x_end, vec_grad_end, color=GREEN)
        para_line2 = DashedLine(vec_y_end, vec_grad_end, color=RED)
        
        self.play(Create(para_line1), Create(para_line2))
        
        grad_vector = Arrow(
            start=axes.c2p(x0, y0, z0),
            end=vec_grad_end,
            color=YELLOW, buff=0, stroke_width=4
        )
        
        self.play(GrowArrow(grad_vector))
        self.wait()
        
        self.play(
            FadeOut(step3_text),
            FadeOut(para_line1), FadeOut(para_line2),
            FadeOut(arrow_x_3d), FadeOut(arrow_y_3d),
            FadeOut(grad_vector),
            FadeOut(dot_p)
        )
        
        # --- PART 4: DYNAMIC MOVEMENT ---
        
        final_text = Tex("Moving along the curve...", color=WHITE).scale(0.8)
        # Anchor to Bottom Left like other steps (consistency)
        final_text.to_corner(DL).shift(UP * 1.5 + RIGHT * 0.5)
        self.add_fixed_in_frame_mobjects(final_text)

        # FADE IN TRACKERS HERE
        self.play(
            val_label_x.animate.set_opacity(1),
            val_label_y.animate.set_opacity(1),
            val_num_x.animate.set_opacity(1),
            val_num_y.animate.set_opacity(1),
            run_time=1
        )

        t_tracker = ValueTracker(0)

        # --- UPDATERS ---
        def update_x_val(m):
            t = t_tracker.get_value()
            curr_x = 2 * np.cos(t)
            m.set_value(-0.6 * curr_x)
            # Buffer ensures no overlap with equal sign
            m.next_to(val_label_x, RIGHT, buff=0.5)
            self.add_fixed_in_frame_mobjects(m)
            
        def update_y_val(m):
            t = t_tracker.get_value()
            curr_y = 2 * np.sin(t)
            m.set_value(-0.6 * curr_y)
            m.next_to(val_label_y, RIGHT, buff=0.5)
            self.add_fixed_in_frame_mobjects(m)

        val_num_x.add_updater(update_x_val)
        val_num_y.add_updater(update_y_val)

        def get_dynamic_vectors():
            t = t_tracker.get_value()
            curr_x = 2 * np.cos(t)
            curr_y = 2 * np.sin(t)
            curr_z = func(curr_x, curr_y)
            
            dx = -0.6 * curr_x
            dy = -0.6 * curr_y
            
            origin_pt = axes.c2p(curr_x, curr_y, curr_z)
            
            dz_x = dx * (-0.6 * curr_x)
            dz_y = dy * (-0.6 * curr_y)
            dz_grad = dz_x + dz_y
            
            arrow_x = Arrow(
                start=origin_pt,
                end=axes.c2p(curr_x + dx, curr_y, curr_z + dz_x),
                color=RED, buff=0, stroke_width=3
            )
            
            arrow_y = Arrow(
                start=origin_pt,
                end=axes.c2p(curr_x, curr_y + dy, curr_z + dz_y),
                color=GREEN, buff=0, stroke_width=3
            )
            
            arrow_grad = Arrow(
                start=origin_pt,
                end=axes.c2p(curr_x + dx, curr_y + dy, curr_z + dz_grad),
                color=YELLOW, buff=0, stroke_width=5
            )
            
            surf_dot = Dot3D(point=origin_pt, color=YELLOW)
            
            return VGroup(arrow_x, arrow_y, arrow_grad, surf_dot)

        dynamic_group = always_redraw(get_dynamic_vectors)
        self.add(dynamic_group)
        
        self.play(t_tracker.animate.set_value(2 * PI), run_time=6, rate_func=linear)
        self.wait()

%manim -qk -v warning GradientConcept